# Bengali LLM — Colab Inference Server
--------------------------------------
Run these cells in order to turn this Colab instance into an ngrok-backed API server for the fine-tuned Qwen 2.5-3B model.

In [ ]:
!pip install -q \
    transformers==4.46.3 \
    peft==0.13.2 \
    bitsandbytes==0.44.1 \
    accelerate==1.1.1 \
    fastapi==0.115.4 \
    uvicorn==0.32.0 \
    pyngrok==7.2.2 \
    nest_asyncio==1.6.0

In [ ]:
import torch
import nest_asyncio
import uvicorn
import threading
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import List, Dict, Optional
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

nest_asyncio.apply()   # allows uvicorn inside Colab's event loop

In [ ]:
BASE_MODEL_ID  = "Qwen/Qwen2.5-3B-Instruct"
ADAPTER_PATH   = "/content/drive/MyDrive/your_lora_adapter_folder"   # ← update this

# 4-bit quantisation config (same as training)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

print("Loading base model in 4-bit...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

print("Loading LoRA adapter...")
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()
print("✅ Model ready!")

In [ ]:
SYSTEM_PROMPT = (
    "You are a helpful Bengali language assistant developed by the "
    "Calcutta University Data Science Lab. You are fine-tuned on Bengali "
    "datasets to understand and generate Bengali text accurately. "
    "Always respond in the same language as the user's question. "
    "Be concise, factual, and helpful."
)

In [ ]:
def run_inference(
    question: str,
    conversation_history: List[Dict[str, str]],
    max_new_tokens: int = 512,
    temperature: float = 0.7,
    top_p: float = 0.9,
) -> str:
    """
    Build the full chat prompt using Qwen's chat template,
    run inference, return only the new assistant text.
    """
    # Build messages list: system → history → current question
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    messages.extend(conversation_history)          # previous turns (from our backend)
    messages.append({"role": "user", "content": question})

    # Apply Qwen chat template (handles special tokens automatically)
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    # Decode only the newly generated tokens (strip the prompt)
    input_length = model_inputs["input_ids"].shape[1]
    new_tokens = output_ids[0][input_length:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    return response

In [ ]:
app = FastAPI(title="Bengali LLM Colab Inference")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)


class InferenceRequest(BaseModel):
    question: str
    conversation_history: Optional[List[Dict[str, str]]] = []
    max_new_tokens: Optional[int] = 512
    temperature: Optional[float] = 0.7


class InferenceResponse(BaseModel):
    response: str


@app.get("/health")
def health():
    return {"status": "ok", "model": "Qwen2.5-3B Bengali Fine-tuned"}


@app.post("/generate", response_model=InferenceResponse)
def generate(req: InferenceRequest):
    try:
        result = run_inference(
            question=req.question,
            conversation_history=req.conversation_history or [],
            max_new_tokens=req.max_new_tokens or 512,
            temperature=req.temperature or 0.7,
        )
        return InferenceResponse(response=result)
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))

In [ ]:
from pyngrok import ngrok, conf

# Set your ngrok auth token (free account works, static domain needs paid plan)
NGROK_AUTH_TOKEN = "your_ngrok_auth_token_here"    # ← from ngrok dashboard
NGROK_STATIC_DOMAIN = "your-static-domain.ngrok-free.app"  # ← from ngrok dashboard

conf.get_default().auth_token = NGROK_AUTH_TOKEN

# Open tunnel on port 8001 (avoid conflict with local dev)
listener = ngrok.connect(
    addr=8001,
    domain=NGROK_STATIC_DOMAIN,   # static domain → URL never changes
)
print(f"✅ Public URL: {listener.public_url}")
print(f"   Endpoint for backend .env:  {listener.public_url}/generate")

# Run FastAPI in a background thread so the cell doesn't block
def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8001, log_level="warning")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
print("🚀 Inference server running!")